# Industrial Anomaly Detection on MVTec AD (Phase 8)

Runs in Google Colab. All real MVTec processing happens here, never on a
laptop. Calls `evat.anomaly.*` / `evat.data.datasets.mvtec` (Phase 1,
unmodified) / `evat.features.encoders` (Phase 5, unmodified) — does not
reimplement the dataset adapter or CNN encoder.

**MVTec AD is used separately from the YouTube-VOS video pipeline.**
It is a still-image dataset, not treated as video here.

License: CC BY-NC-SA 4.0, non-commercial use only. MVTec AD requires
completing MVTec's official registration form before download — this
notebook does not bypass that. See `docs/datasets.md`.

In [ ]:
%pip install -q -e .

In [ ]:
import os
from pathlib import Path

os.environ["EVAT_DATA_ROOT"] = "/content/data"  # example only; set to the real path
dataset_root = Path(os.environ["EVAT_DATA_ROOT"]) / "mvtec_ad"

## Verify dataset and print statistics

In [ ]:
from collections import Counter

from evat.data.datasets.mvtec import build_manifest, discover_categories
from evat.data.validation import validate_manifest

categories = discover_categories(dataset_root)
print("categories:", categories)

records = build_manifest(dataset_root, dataset_version="<fill in verified MVTec version/date>")
report = validate_manifest(records, dataset_root=dataset_root)
print(report.summary())

print("by split:", Counter(r.split for r in records))
print("by label:", Counter(r.label for r in records))
print("with annotation (defects):", sum(1 for r in records if r.annotation_path))

## SMOKE TEST (one category, tiny subset)

Verify data loading, feature extraction, model fitting, scoring, and
visualization before running the full baseline.

In [ ]:
import torch

from evat.anomaly.config import AnomalyConfig
from evat.anomaly.dataset import filter_records, load_mvtec_image, load_mvtec_mask
from evat.anomaly.localization import upsample_anomaly_map
from evat.anomaly.model import anomaly_map, fit_category_anomaly_model, image_anomaly_score
from evat.anomaly.threshold import derive_threshold_from_normal_scores
from evat.features.encoders import CNNEncoderConfig, CNNFeatureEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

config = AnomalyConfig.from_yaml("configs/anomaly.yaml")
size = (config.input_height, config.input_width)
encoder = CNNFeatureEncoder(CNNEncoderConfig(pretrained=True, frozen=config.frozen)).to(device)

smoke_category = categories[0]
train_records = filter_records(records, smoke_category, "train", label="good")[:10]
train_images = torch.stack(
    [load_mvtec_image(dataset_root, r.image_path, size) for r in train_records]
).to(device)

model = fit_category_anomaly_model(smoke_category, encoder, train_images, eps=config.covariance_eps)

test_records = filter_records(records, smoke_category, "test")[:10]
test_images = torch.stack(
    [load_mvtec_image(dataset_root, r.image_path, size) for r in test_records]
).to(device)
scores = image_anomaly_score(model, encoder, test_images)
print("smoke scores:", scores)

train_scores = image_anomaly_score(model, encoder, train_images)
threshold = derive_threshold_from_normal_scores(train_scores, config.threshold_percentile)
print("threshold (from training data only):", threshold)

## BASELINE RUN — per category

Fit and evaluate one normality model PER category (never a single global
model — see docs/anomaly_task_definition.md). If a category's full test
set doesn't fit within session limits, process categories one at a time
(already the loop structure below) rather than silently subsampling.

In [ ]:
import subprocess
import time

import numpy as np

from evat.anomaly.metrics import AnomalyMetrics, pr_auc_score, roc_auc_score
from evat.anomaly.record import save_anomaly_result

git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
hardware = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

for category in categories:
    start = time.time()

    train_records = filter_records(records, category, "train", label="good")
    train_images = torch.stack(
        [load_mvtec_image(dataset_root, r.image_path, size) for r in train_records]
    ).to(device)
    model = fit_category_anomaly_model(category, encoder, train_images, eps=config.covariance_eps)

    test_records = filter_records(records, category, "test")
    test_images = torch.stack(
        [load_mvtec_image(dataset_root, r.image_path, size) for r in test_records]
    ).to(device)
    scores = image_anomaly_score(model, encoder, test_images)
    labels = np.array([0 if r.label == "good" else 1 for r in test_records])

    image_metrics = AnomalyMetrics(
        roc_auc=roc_auc_score(scores, labels), pr_auc=pr_auc_score(scores, labels)
    )

    # Pixel-level ROC-AUC over anomalous test images only (normal images have no mask).
    pixel_scores, pixel_labels = [], []
    for r in test_records:
        if r.annotation_path is None:
            continue
        img = load_mvtec_image(dataset_root, r.image_path, size).unsqueeze(0).to(device)
        amap = anomaly_map(model, encoder, img)
        upsampled = upsample_anomaly_map(amap, size=size)
        gt_mask = load_mvtec_mask(dataset_root, r.annotation_path, size=size)
        pixel_scores.append(upsampled.flatten())
        pixel_labels.append(gt_mask.flatten())

    pixel_roc_auc = None
    if pixel_scores:
        pixel_roc_auc = roc_auc_score(np.concatenate(pixel_scores), np.concatenate(pixel_labels))

    runtime_seconds = time.time() - start
    print(category, image_metrics, "pixel_roc_auc:", pixel_roc_auc, "runtime:", runtime_seconds)

    experiment_config = {
        "anomaly": config.__dict__,
        "num_train": len(train_records),
        "num_test": len(test_records),
    }
    save_anomaly_result(
        "results/phase8",
        category,
        config=experiment_config,
        metrics=image_metrics,
        git_commit=git_commit,
        dataset_version="<fill in verified MVTec version/date>",
        hardware=hardware,
        runtime_seconds=runtime_seconds,
        pixel_roc_auc=pixel_roc_auc,
    )

## Qualitative visualization

Include true normal, clear anomaly, subtle anomaly, and any localization
failure — not only successful examples.

In [ ]:
from evat.visualization.anomaly_overlay import make_anomaly_panel

os.makedirs("results/phase8/qualitative", exist_ok=True)
for r in test_records[:6]:
    img_tensor = load_mvtec_image(dataset_root, r.image_path, size)
    amap = anomaly_map(model, encoder, img_tensor.unsqueeze(0).to(device))
    upsampled = upsample_anomaly_map(amap, size=size)
    gt_mask = (
        load_mvtec_mask(dataset_root, r.annotation_path, size=size) if r.annotation_path else None
    )
    panel = make_anomaly_panel(img_tensor.numpy(), upsampled, gt_mask)
    panel.save(f"results/phase8/qualitative/{category}_{r.sample_id.replace('/', '_')}.png")

## Save results

Update `docs/experiments.md` with the actual printed statistics/metrics/
runtime above. Do not hand-edit numbers not produced by this notebook.